# Install & Import

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import optuna
import lightgbm as lgbm
from optuna.integration import LightGBMPruningCallback
from optuna.samplers import TPESampler
from optuna.pruners import HyperbandPruner
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    make_scorer,
    precision_recall_curve,
    precision_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
    f1_score,
    recall_score,
    average_precision_score,
    matthews_corrcoef,
    balanced_accuracy_score,
)

# Settings/Constant variables

In [ ]:
RND_STATE = 37
TRAIN_TEST_SPLIT = 0.07
TRAIN_VALID_SPLIT = 0.07
EARLY_STOPPING_ROUNDS = 50
HYPERPARAM_TUNING_NFOLDS = 3
TERMINATION_OPTUNA = 20 * 60  # IN SECONDS
DATASET_FILENAME = "cls_default_modeling.parquet"

In [ ]:
BEST_PARAMS = {
    "seed": 37,
    "device_type": "gpu",
    "objective": "binary",
    "is_unbalance": True,
    "metric": "average_precision",
    "n_estimators": 508,
    "learning_rate": 0.012877525071658265,
    "num_leaves": 127,
    "min_data_in_leaf": 171,
    "max_depth": 24,
    "feature_fraction": 0.21429130841544336,
    "bagging_fraction": 0.9761168922030805,
    "bagging_freq": 16,
    "max_bin": 103,
    "lambda_l2": 84.87400986645069,
    "lambda_l1": 22.70916655448999,
    "min_sum_hessian_in_leaf": 0.1994630469325395,
    "min_gain_to_split": 8.585454548895282,
    "feature_fraction_bynode": 0.5418218276374791,
    "path_smooth": 18.66700249343249,
    "extra_trees": False,
}

# Read data

In [ ]:
SCORERS = {
    "f1": make_scorer(f1_score),
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "roc_auc": make_scorer(roc_auc_score, needs_proba=True),
    "pr_auc": make_scorer(average_precision_score, needs_proba=True),
    "mcc": make_scorer(matthews_corrcoef),
    "balanced_accuracy": make_scorer(balanced_accuracy_score),
}

In [ ]:
modeling = pd.read_parquet(f"data/modeling/{DATASET_FILENAME}")
modeling.drop(columns=["weight", "target"], inplace=True)

In [ ]:
float_columns = modeling.select_dtypes(include=["float64", "float32"]).columns
int_columns = modeling.select_dtypes(include=["int64", "int32", "int"]).columns

modeling[float_columns] = modeling[float_columns].astype("float32")
modeling[int_columns] = modeling[int_columns].astype("int32")

In [ ]:
worth = modeling[modeling["worthy"] == 1]
not_worth = modeling[modeling["worthy"] == 0]

modeling["worthy"].hist()
print(
    f"Observations with worthy class: {len(worth)} ({len(worth)/modeling.shape[0]*100:.2f}%)"
)

In [ ]:
X = modeling.drop(columns=["worthy"])
y = modeling["worthy"]

# Train/test split

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=TRAIN_TEST_SPLIT, stratify=y, random_state=RND_STATE
)

X_train_full.shape, X_test.shape, y_train_full.shape, y_test.shape

## Train/validation split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=TRAIN_VALID_SPLIT,
    stratify=y_train_full,
    random_state=RND_STATE,
)

# Fit default model

In [ ]:
MODEL_STATIC_PARAMS = {
    "seed": RND_STATE,
    "device_type": "gpu",
    "objective": "binary",
    # "metric": "binary_logloss",
    "is_unbalance": True,
    "metric": "average_precision",
}

In [ ]:
default_lgbm_model = lgbm.LGBMClassifier(**MODEL_STATIC_PARAMS)
default_lgbm_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgbm.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=True)
    ],
)

model_path = "models/default_cls_model.joblib"
joblib.dump(default_lgbm_model, model_path)

In [ ]:
# Predict on the test set
y_pred = default_lgbm_model.predict(X_test)
y_pred_proba = default_lgbm_model.predict_proba(X_test)[:, 1]

In [ ]:
# Calculate evaluation metrics
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_pred_proba)
pr_auc = average_precision_score(y_test, y_pred_proba)
mcc = matthews_corrcoef(y_test, y_pred)
balanced_acc = balanced_accuracy_score(y_test, y_pred)

# Create a dictionary with the evaluation metrics and an additional 'Type' column
results = {
    "Type": ["default"],
    "F1 Score": [f1],
    "MCC": [mcc],
    "PR AUC": [pr_auc],
    "Balanced Accuracy": [balanced_acc],
    "Precision": [precision],
    "Recall": [recall],
    "AUC": [roc_auc],
}

DF_SCORES = pd.DataFrame(results)
print(DF_SCORES)

In [ ]:
# Display confusion matrix and classification report
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Plot confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("Confusion Matrix")
plt.show()

# Plot ROC curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
roc_auc_curve = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, lw=2, label=f"ROC curve (area = {roc_auc_curve:.2f})")
plt.plot([0, 1], [0, 1], lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.show()

# Plot LightGBM feature importance (by split count)
lgbm.plot_importance(
    default_lgbm_model.booster_, max_num_features=30, importance_type="split"
)
plt.title("Top 30 Most Important Features (LightGBM)")
plt.show()

# Optuna hiperparameter optimization

In [ ]:
opt_cv = StratifiedKFold(
    n_splits=HYPERPARAM_TUNING_NFOLDS, shuffle=True, random_state=RND_STATE
)

## Sampler and pruner definition

In [ ]:
sampler = TPESampler(
    seed=RND_STATE,
    multivariate=True,
    group=True,
    n_startup_trials=25,
    constant_liar=True,
)

pruner = HyperbandPruner(min_resource=100, max_resource=2_000, reduction_factor=3)

## Define objective

In [ ]:
def objective(trial):
    params = {
        **MODEL_STATIC_PARAMS,
        "objective": "binary",
        "metric": "average_precision",
        "n_estimators": trial.suggest_int("n_estimators", 100, 2_000),
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.5, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 8, 128),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 2, 200),
        "max_depth": trial.suggest_int("max_depth", 2, 64),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.1, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.1, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 0, 25),
        "max_bin": trial.suggest_int("max_bin", 64, 1024),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 100.0),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 100.0),
        "min_sum_hessian_in_leaf": trial.suggest_float(
            "min_sum_hessian_in_leaf", 1e-4, 100.0, log=True
        ),
        "min_gain_to_split": trial.suggest_float("min_gain_to_split", 0.0, 10.0),
        "feature_fraction_bynode": trial.suggest_float(
            "feature_fraction_bynode", 0.1, 1.0
        ),
        "path_smooth": trial.suggest_float("path_smooth", 0.0, 100.0),
        "extra_trees": trial.suggest_categorical("extra_trees", [True, False]),
        "verbosity": -1,
    }

    dtrain = lgbm.Dataset(X_train_full, label=y_train_full)

    folds = list(opt_cv.split(X_train_full, y_train_full))

    try:
        cv_res = lgbm.cv(
            params,
            dtrain,
            folds=folds,
            num_boost_round=int(params["n_estimators"]),
            callbacks=[
                lgbm.early_stopping(
                    stopping_rounds=EARLY_STOPPING_ROUNDS, first_metric_only=True
                ),
                LightGBMPruningCallback(trial, "average_precision", report_interval=1),
            ],
            seed=RND_STATE,
        )
    except lgbm.basic.LightGBMError:
        raise optuna.TrialPruned()

    key = [k for k in cv_res.keys() if "average_precision-mean" in k][0]
    return float(max(cv_res[key]))

## Study

In [ ]:
# Run Optuna search (TPE + ASHA)
study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)

### Inject prev params

In [ ]:
default_params = {
    **MODEL_STATIC_PARAMS,
    "n_estimators": 100,
    "learning_rate": 0.1,
    "num_leaves": 31,
    "max_depth": -1,
    "min_data_in_leaf": 20,
    "feature_fraction": 1.0,
    "bagging_fraction": 1.0,
    "bagging_freq": 0,
    "lambda_l1": 0.0,
    "lambda_l2": 0.0,
    "min_gain_to_split": 0.0,
    "min_sum_hessian_in_leaf": 1e-3,
    "max_bin": 255,
    "feature_fraction_bynode": 1.0,
    "path_smooth": 0,
    "extra_trees": False,
}

In [ ]:
search_space = {
    "n_estimators",
    "learning_rate",
    "num_leaves",
    "max_depth",
    "min_data_in_leaf",
    "feature_fraction",
    "bagging_fraction",
    "bagging_freq",
    "lambda_l1",
    "lambda_l2",
    "min_gain_to_split",
    "tweedie_variance_power",
    "alpha",
    "fair_c",
    "min_sum_hessian_in_leaf",
    "max_bin",
    "feature_fraction_bynode",
    "path_smooth",
    "extra_trees",
}

injected_params = {k: v for k, v in BEST_PARAMS.items() if k in search_space}

# Add conditional keys only when sensowne dla celu
if injected_params.get("objective") != "tweedie":
    injected_params.pop("tweedie_variance_power", None)
if injected_params.get("objective") != "huber":
    injected_params.pop("alpha", None)
if injected_params.get("objective") != "fair":
    injected_params.pop("fair_c", None)

if injected_params:
    study.enqueue_trial(injected_params)

study.enqueue_trial(default_params)

### Run study

In [ ]:
study.optimize(objective, timeout=TERMINATION_OPTUNA, show_progress_bar=True)

In [ ]:
BEST_PARAMS = study.best_trial.params
BEST_PARAMS = {**MODEL_STATIC_PARAMS, **study.best_trial.params}
print("Best params:", BEST_PARAMS)

## Fit optimized model

In [ ]:
opt_lgbm_model = lgbm.LGBMClassifier(**BEST_PARAMS)
opt_lgbm_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgbm.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=True)
    ],
)
os.makedirs("models", exist_ok=True)
joblib.dump(opt_lgbm_model, "models/opt_cls_model.joblib")

## Analyse opimal model

In [ ]:
# Evaluate & log into DF_SCORES
y_pred = opt_lgbm_model.predict(X_test)
y_proba = opt_lgbm_model.predict_proba(X_test)[:, 1]

In [ ]:
# Calculate evaluation metrics
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)
mcc = matthews_corrcoef(y_test, y_pred)
balanced_acc = balanced_accuracy_score(y_test, y_pred)

# Create a dictionary with the evaluation metrics and an additional 'Type' column
results = {
    "Type": ["opt"],
    "F1 Score": [f1],
    "MCC": [mcc],
    "PR AUC": [pr_auc],
    "Balanced Accuracy": [balanced_acc],
    "Precision": [precision],
    "Recall": [recall],
    "AUC": [roc_auc],
}

DF_SCORES = pd.DataFrame(results)
print(DF_SCORES)

In [ ]:
# Display confusion matrix and classification report
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Plot confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("Confusion Matrix")
plt.show()

# Plot ROC curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc_curve = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, lw=2, label=f"ROC curve (area = {roc_auc_curve:.2f})")
plt.plot([0, 1], [0, 1], lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.show()

# Plot LightGBM feature importance (by split count)
lgbm.plot_importance(
    opt_lgbm_model.booster_, max_num_features=30, importance_type="split"
)
plt.title("Top 30 Most Important Features (LightGBM)")
plt.show()

# Final model

In [ ]:
X_final_train, X_final_test, y_final_train, y_final_test = train_test_split(
    X, y, test_size=0.001, stratify=y, random_state=RND_STATE
)

X_tr_f, X_val_f, y_tr_f, y_val_f = train_test_split(
    X_final_train,
    y_final_train,
    test_size=TRAIN_VALID_SPLIT,
    stratify=y_final_train,
    random_state=RND_STATE,
)

## Train

In [ ]:
if BEST_PARAMS is not None:
    final_lgbm_model = lgbm.LGBMClassifier(**BEST_PARAMS, verbosity=-1)
else:
    final_lgbm_model = lgbm.LGBMClassifier(**MODEL_STATIC_PARAMS, verbosity=-1)

final_lgbm_model.fit(
    X_tr_f,
    y_tr_f,
    eval_set=[(X_val_f, y_val_f)],
    callbacks=[
        lgbm.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=True)
    ],
)
joblib.dump(final_lgbm_model, "models/final_cls_model.joblib")

In [ ]:
# Predict on the test set
y_pred = final_lgbm_model.predict(X_final_test)
y_pred_proba = final_lgbm_model.predict_proba(X_final_test)[:, 1]

# Calculate evaluation metrics
f1 = f1_score(y_final_test, y_pred)
precision = precision_score(y_final_test, y_pred, zero_division=0)
recall = recall_score(y_final_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_final_test, y_pred_proba)
pr_auc = average_precision_score(y_final_test, y_pred_proba)
mcc = matthews_corrcoef(y_final_test, y_pred)
balanced_acc = balanced_accuracy_score(y_final_test, y_pred)

# Print the metrics
print("SCORES:")
print(f"F1 Score:          {f1:.4f}")
print(f"MCC:               {mcc:.4f}")
print(f"PR AUC:            {pr_auc:.4f}")
print(f"Balanced Accuracy: {balanced_acc:.4f}")
print(f"Precision:         {precision:.4f}")
print(f"Recall:            {recall:.4f}")
print(f"AUC:               {roc_auc:.4f}")

# Display confusion matrix and classification report
cm = confusion_matrix(y_final_test, y_pred)
print("\nConfusion Matrix:")
print(cm)
print("\nClassification Report:")
print(classification_report(y_final_test, y_pred))

# Plot confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("Confusion Matrix")
plt.show()

# Plot ROC curve
fpr, tpr, _ = roc_curve(y_final_test, y_pred_proba)
roc_auc_curve = auc(fpr, tpr)

plt.figure()
plt.plot(
    fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (area = {roc_auc_curve:.2f})"
)
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Plot LightGBM feature importance (by split count)
lgbm.plot_importance(
    final_lgbm_model.booster_, max_num_features=30, importance_type="split"
)
plt.title("Top 30 Most Important Features (LightGBM)")
plt.show()